In [1]:
from COMPILER import compile_microcode, compile_code

In [2]:
compile_microcode()

Saved in microcode.raw


In [12]:
code = """
; Segments initialization
seg:
    MOV DS 0x4000    ;Data segment
    MOV SS 0x8000    ;Stack segment
    MOV SP 0x4000    ;Stack pointer, point at the top of the stack
    JMP start

f:                     ; f -> a + b - 1
    PUSH BP
    MOV BP SP
    SUB SP 1           ; def x
    
    MOV [BP+0xFFFF] 1  ; x = 1
    MOV AX [BP+3]      ; fetch a
    MOV BX [BP+2]      ; fetch b

    ADD AX BX          ; a = a + b
    SUB AX [BP+0xFFFF] ; a = a - 1

    MOV SP BP
    POP BP
    RET
    

start:
    MOV AX [0]
    MOV BX [1]
    
    PUSH AX     ; a = 5
    PUSH BX     ; b = 2
    CALL f
    ADD SP 2

    MOV [2] AX
    JMP end

end:
    HLT

"""
compile_code(code)

'808 f 4000 808 10 8000 808 12 4000 440 2c 489 13 809 13 12 888 12 1 820 13 1 ffff 80c a 13 3 80c b 13 2 849 a b 88c a 13 ffff 809 12 13 491 13 1 80a a 0 80a b 1 489 a 489 b 498 b 848 12 2 811 2 a 440 40 fff'

In [13]:
code = """
seg:                 ;Segments initialization
    MOV DS 0x4000    ;Data segment
    MOV SS 0x8000    ;Stack segment
    MOV SP 0x4000    ;Stack pointer, point at the top of the stack
    JMP start

f:                   ;f -> a × b - 1
    PUSH BP
    MOV BP SP
    
    MOV T1 8         ;Cycle eight times for 8-bit multiplication
    MOV AX [BP+3]    ;Fetch a
    MOV BX [BP+2]    ;Fetch b
    MOV CX 0         ;Clear result
    
L1:
    MOV T2 BX        ;Determine the lowest position of b
    AND T2 1
    JZ L0
    ADD CX AX

L0:
    SHL AX
    SHR BX
    DEC T1
    JNZ L1
    
    MOV SP BP
    POP BP
    RET

start:
    MOV AX [0]       ;a = [0]
    PUSH AX
    
    MOV BX [1]       ;b = [1]
    PUSH BX
        
    CALL f           ;[2] = a × b
    ADD SP 2
    MOV [2] CX
    
    JMP end

end:
    HLT
"""

compile_code(code)

'808 f 4000 808 10 8000 808 12 4000 440 37 489 13 809 13 12 808 16 8 80c a 13 3 80c b 13 2 808 c 0 809 17 b 908 17 1 448 29 849 c a 419 a 421 b 409 16 450 1e 809 12 13 491 13 1 80a a 0 489 a 80a b 1 489 b 498 b 848 12 2 811 2 c 440 4b fff'